# 01 · Augmentation Anatomy

**What does each augmentation transform actually *do* to a training example?** This
notebook takes the standard music-source-separation recipe apart, one transform at a
time, and shows — in pictures and numbers — what remix, gain, and sign flip change
about the signal the network sees. It is the visual companion to
[`../THEORY.md`](../THEORY.md) §1–3 and sets up the experiments in
[`02_factorization_scaling_experiments.ipynb`](02_factorization_scaling_experiments.ipynb).

Nothing here needs a GPU. A few cells that need the decoded MUSDB shards (rendering
real audio, the subset-balance report) are marked **⚠️ RUN THIS LATER**; everything
else runs on CPU from synthetic signals, but the notebook ships **un-executed** so the
committed file is a clean scaffold.

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §5 (the exact transforms), §3.1
  (the leave-one-out design), §3.5 (per-transform RNG streams); [`../THEORY.md`](../THEORY.md)
  §1 (remix as a product of marginals), §2 (effective sample size), §3 (gain & flip).
- **Data prep is *not* repeated here.** Acquisition, licensing, the 86/14/50 split, and
  the STFT/chunking front end live in Direction 01's notebooks
  ([`../../01-loss-function-study/notebooks/01_data_and_eda.ipynb`](../../01-loss-function-study/notebooks/01_data_and_eda.ipynb)
  and `02_pipeline_and_model.ipynb`). This direction reuses them verbatim.
- **Code, not prose, is authoritative:** every transform below is
  `singnet/data/augment.py`; the sampler is `singnet/data/musdb_dataset.py`. If they
  change, this notebook changes with them.

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU. Installs the pinned env and mounts Drive.
# Skip locally if you already `pip install -r requirements.txt`.
#
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"] = "/content/drive/MyDrive/musdb_shards"
print("Bootstrap cell — run on Colab only (see comments). No-op here.")

## 1 · The recipe we are taking apart

Three transforms, applied per 6 s chunk in the waveform domain, in this fixed order
(remix → gain → flip). The constants are code-verified against Open-Unmix and Demucs
(MASTER_PLAN §5); the leave-one-out study measures what each is worth.

| Transform | Operation | Constants | Provenance |
|---|---|---|---|
| **remix** | vocals from track *i*, accompaniment from an independent track *j* | — | UMX `random_track_mix`; Demucs `Remix` |
| **gain** | each source × `U(0.25, 1.25)` | 0.25–1.25 linear | UMX `_augment_gain`; Demucs `Scale` |
| **flip** | each source `s → −s` with `p = 0.5` | p = 0.5 | Demucs `FlipSign` |

Channel-swap and pitch/tempo are deliberately out of scope (mono pipeline; heavyweight
separate path) — documented in MASTER_PLAN §2's scope note, not silently dropped. The
mixture is always the **sum** of the (possibly transformed) sources, so additivity is
exact in every arm.

## 2 · A synthetic sandbox

To *see* each transform without touching the dataset, we build a tiny in-memory store
of tone-plus-noise "tracks" and run them through the real `AugmentPipeline`. This is
the same object training uses — we are visualizing production code, not a toy
re-implementation. (CPU; left un-executed in the committed file.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from singnet.audio import STFT
from singnet.data import AugmentPipeline, InMemoryStore, MusdbChunks, Manifest
import pandas as pd

SR = 44100

def tone(freq, secs=6.5, amp=0.3):
    t = np.arange(int(secs * SR)) / SR
    return (amp * np.sin(2 * np.pi * freq * t)).astype(np.float32)

# four synthetic "songs" with distinct vocal/accompaniment content
rng = np.random.default_rng(0)
tracks = {}
for i in range(4):
    v = tone(220 + 50 * i) + 0.01 * rng.standard_normal(int(6.5 * SR)).astype(np.float32)
    a = tone(90 + 20 * i) + 0.5 * tone(300 + 40 * i) + 0.01 * rng.standard_normal(int(6.5 * SR)).astype(np.float32)
    tracks[f"song_{i:02d}"] = {"vocals": v, "accompaniment": a}
store = InMemoryStore(tracks, SR)
manifest = Manifest(pd.DataFrame([{"track": t, "split": "train"} for t in tracks]))
stft = STFT()
print("sandbox ready:", store.track_names())

### 2.1 Gain — before/after

*Figure to render:* the vocal chunk before and after per-source gain, as a waveform and
a log-magnitude spectrogram. **How to read it:** gain rescales the amplitude (the
waveform's envelope shrinks/grows) but does not move energy between frequencies. Because
the model's input is *per-chunk standardized* `log(1+|X|)`, the network is nearly blind
to a *common* scale — the informative part of gain is the change in the **vocal-to-
accompaniment balance** (THEORY §3.2).

In [ ]:
# Gain on its own stream (remix/flip off) so we isolate its effect. CPU; render later.
gain_pipe = AugmentPipeline(remix=False, gain=True, flip=False, seed=0)
base = {"vocals": tracks["song_00"]["vocals"][:STFT().n_fft * 60].copy()}
out = gain_pipe.apply_gain(base, step=3)
print("gain factor ~", float(np.max(np.abs(out['vocals'])) / np.max(np.abs(base['vocals']))))
# fig, ax = plt.subplots(2, 2, figsize=(10, 5))  # waveform + |STFT| before/after
# ... plot base['vocals'] vs out['vocals'] and their |STFT|

### 2.2 Sign flip — the invisible transform

*Figure to render:* the waveform before/after a sign flip (a mirror image) beside their
log-magnitude spectrograms (**identical**). **How to read it:** the whole point is that
the two spectrograms are the same to machine precision — flipping polarity leaves every
magnitude untouched. For a magnitude-masking model this transform changes *nothing*,
which is exactly why its leave-one-out delta is the design's negative control (§3, and
the numeric check in section 3 below).

In [ ]:
# Flip is |STFT|-invariant: |STFT(-s)| == |STFT(s)|. CPU; render later.
import torch
s = torch.from_numpy(tracks["song_00"]["vocals"][:STFT().n_fft * 60].copy())
mag_pos = STFT().transform(s.unsqueeze(0)).abs()      # |STFT(s)|
mag_neg = STFT().transform((-s).unsqueeze(0)).abs()   # |STFT(-s)|
print("max |STFT| difference under sign flip:", float((mag_pos - mag_neg).abs().max()))
# fig: waveform s vs -s (mirror) | |STFT(s)| vs |STFT(-s)| (identical) | their difference (~0)

### 2.3 Remix — musically incoherent but statistically rich

*Figure to render:* a **coherent** mixture (vocals + accompaniment from the *same* song)
next to a **remixed** mixture (vocals from one song, accompaniment from another), as
spectrograms. **How to read it:** the remixed mixture is a valid training example whose
marginals match real music but whose vocal/accompaniment coupling is destroyed
(THEORY §1). It samples a *larger* mixture manifold — the mechanism behind the
pre-registered bet that remix is the top small-data lever — at the cost of some
musically incoherent combinations the test set never contains (the honest caveat).

In [ ]:
# Coherent vs remixed mixture from the real sampler (remix off vs on). CPU; render later.
coherent = MusdbChunks(store, manifest, "train", seed=0, length=8,
                       pipeline=AugmentPipeline(remix=False, gain=False, flip=False))
remixed = MusdbChunks(store, manifest, "train", seed=0, length=8,
                      pipeline=AugmentPipeline(remix=True, gain=False, flip=False))
c0, r0 = coherent[0], remixed[0]
print("coherent vocals track == accompaniment track (same song); remixed draws differ.")
print("mixture == vocals + accompaniment holds in both:",
      bool(torch.allclose(c0["mixture"], c0["vocals"] + c0["accompaniment"], atol=1e-6)))
# fig: |STFT(coherent mixture)| vs |STFT(remixed mixture)|

## 3 · The flip-invariance sanity argument, made concrete

THEORY §3.1 proves `|STFT(−s)| = |STFT(s)|` from the linearity of the STFT, and argues
that — with negation-symmetric source marginals and remix on — sign flip is a
measure-preserving map of the training distribution onto itself. The practical
consequence: **Δ_flip ≈ 0 within seed noise is expected, not a failure.** If the
experiment later reports `Δ_flip > σ_seed`, the design's placebo has fired and we stop
and debug (RNG-stream leakage, a normalization fault, or an unintended phase-sensitive
term) *before* trusting any other number (§12, §13).

In [ ]:
# The whole argument in one line: sign flip does not move any magnitude bin.
diff = float((STFT().transform(s.unsqueeze(0)).abs() - STFT().transform((-s).unsqueeze(0)).abs()).abs().max())
print(f"|STFT(-s)| - |STFT(s)| max abs = {diff:.2e}  (≈ machine precision -> Δ_flip is a placebo)")

## 4 · Effective sample size — the combinatorics (THEORY §2)

Remix does not add *songs*; it adds *pairings*. With `N` songs of about `c ≈ 40`
non-overlapping 6 s chunks each, the no-remix pool is `≈ N·c` coupled pairs, while remix
reaches `≈ (N·c)²` distinct mixtures — assembled from only `2·N·c` reused source atoms.
So the effective sample size is bounded, honestly, by `N ≲ n_eff ≲ (N·c)²`, and
"12× more songs ≠ 12× more effective data". The table makes the gap concrete.

In [ ]:
# Pool-size bounds per song-count (THEORY §2). Pure arithmetic; CPU.
import pandas as pd
c = 40  # ~240 s / 6 s non-overlapping chunks per MUSDB track
rows = [{"N_songs": N, "chunks_per_song": c, "no_remix_pool ≈ N·c": N * c,
         "remix_pool ≈ (N·c)^2": (N * c) ** 2, "source_atoms ≈ 2·N·c": 2 * N * c}
        for N in (21, 43, 64, 86)]
combinatorics = pd.DataFrame(rows)
print(combinatorics.to_string(index=False))
# display(combinatorics)  # in a live kernel

## 5 · Subset-balance report (post-G0b)

> ⚠️ **RUN THIS LATER** — needs the *materialized* split manifest · _CPU, ~1 min_

The scaling curve trains on nested subsets `n21 ⊂ n43 ⊂ n64 ⊂ n86` drawn once with
subset-seed 0. `scripts/make_subsets.py` writes the names-only CSVs and prints a
balance report (per-subset song count, and duration / vocal-activity if the
prepare-data index exists) so a pathological draw is visible **before** training. The
pre-registered rule: the seed-0 draw is used regardless — pathologies are reported,
never re-rolled (§4). Run this only after Direction 01's data prep has materialized the
manifest (gate G0b).

In [ ]:
# ⚠️ RUN THIS LATER (CPU) — regenerate + review the nested subsets at gate G0b.
# !python scripts/make_subsets.py \
#     --manifest 01-loss-function-study/configs/splits.csv \
#     --sizes 21 43 64 --seed 0 \
#     --out 02-augmentation-data-scaling/configs/subsets/ \
#     --index $SHARD_ROOT/index.json
#
# Then render the committed subsets + balance here:
# from pathlib import Path
# from singnet.data import load_track_allowlist
# import make_subsets as ms  # scripts/ on sys.path
# subs = {n: load_track_allowlist(f"02-augmentation-data-scaling/configs/subsets/n{n}.csv")
#         for n in (21, 43, 64)}
# display(ms.subset_balance(subs, index=None))
print("Subset-balance report renders after G0b (RUN LATER).")

## 6 · What to listen for

> ⚠️ **RUN THIS LATER** — needs decoded MUSDB shards on Drive · _CPU, ~1 min_

Numbers and spectrograms only go so far — the remix caveat is *audible*. Render a
coherent mixture and a remixed one for the same vocal and listen: the remixed mixture
often clashes in key or tempo ("musically incoherent but statistically rich"). This is
the distribution the network trains on; hearing it builds intuition for why remix both
helps (richer manifold) and risks a train/test mismatch (§1.3).

In [ ]:
# ⚠️ RUN THIS LATER (CPU) — render audio from real shards to listen to the remix caveat.
# from IPython.display import Audio, display
# from singnet.data import WavShardStore, MusdbChunks, AugmentPipeline, Manifest
# store = WavShardStore(os.environ["SHARD_ROOT"])
# manifest = Manifest.from_csv("01-loss-function-study/configs/splits.csv")
# coherent = MusdbChunks(store, manifest, "train", seed=0, length=8,
#                        pipeline=AugmentPipeline(remix=False, gain=False, flip=False))
# remixed  = MusdbChunks(store, manifest, "train", seed=0, length=8,
#                        pipeline=AugmentPipeline(remix=True, gain=False, flip=False))
# display(Audio(coherent[0]["mixture"].numpy(), rate=44100))  # a real song's mixture
# display(Audio(remixed[0]["mixture"].numpy(),  rate=44100))  # a cross-song remix
print("Audio A/B (coherent vs remixed mixture) renders from shards (RUN LATER).")

## 7 · Takeaways

- **remix** re-designs the training distribution (a product of source marginals), moving
  the model onto a strictly larger mixture manifold — the pre-registered top lever.
- **gain** is mostly normalized away on the input; its real content is the vocal/
  accompaniment balance it augments (the target scale).
- **flip** is exactly magnitude-invariant — a placebo, and therefore the design's
  built-in bug detector.
- remix's power is combinatorial in *pairings*, not in new *songs*, which is precisely
  why the scaling curve is an empirical question.

Next: [`02_factorization_scaling_experiments.ipynb`](02_factorization_scaling_experiments.ipynb)
runs the 9 training jobs, measures each transform's leave-one-out Δ against the σ_seed
band, and fits the data-scaling curve.